# Punto 4 — Detección de dedos extendidos en lengua de señas

**Dataset:** El mismo dataset de 27 clases del Punto 3 ([ardamavi/27-class-sign-language-dataset](https://www.kaggle.com/datasets/ardamavi/27-class-sign-language-dataset)).

**Objetivo:** Dada una imagen de mano, predecir **qué dedos están extendidos** usando una red neuronal densa multi-label.

**Formulación:** Clasificación multi-label con 5 salidas binarias:
`[pulgar, índice, medio, anular, meñique]` — 1 = extendido, 0 = flexionado.

**Estrategia de etiquetado:** Se define un mapeo anatómico de cada clase ASL a los dedos extendidos, basado en la descripción visual de cada signo.

## 1. Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    classification_report, confusion_matrix
)
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data/sign_language')

FINGER_NAMES = ['Pulgar', 'Índice', 'Medio', 'Anular', 'Meñique']

print(f'TensorFlow {tf.__version__}')

## 2. Mapeo anatómico: clases ASL → dedos extendidos

Cada fila representa una clase (0–26) y las columnas son:
`[Pulgar, Índice, Medio, Anular, Meñique]`

El mapeo se basa en la configuración estándar de los signos ASL.
Las 27 clases corresponden aproximadamente a A–Z + espacio/número 0.

In [ ]:
# Mapeo usando el NOMBRE de cada clase ASL como clave
# [Pulgar, Índice, Medio, Anular, Meñique] — 1=extendido, 0=flexionado
# Nota: las claves deben coincidir exactamente con CLASS_NAMES del LabelEncoder.
# Si el dataset usa nombres distintos, actualizar las claves aquí.
FINGER_MAP_BY_NAME = {
    'A':     [0, 0, 0, 0, 0],  # puño cerrado, pulgar lateral
    'B':     [0, 1, 1, 1, 1],  # cuatro dedos extendidos, pulgar doblado
    'C':     [1, 1, 1, 1, 1],  # todos curvados formando C
    'D':     [0, 1, 0, 0, 0],  # índice apunta arriba
    'E':     [0, 0, 0, 0, 0],  # todos flexionados
    'F':     [1, 0, 1, 1, 1],  # índice y pulgar se tocan, resto extendidos
    'G':     [1, 1, 0, 0, 0],  # índice y pulgar apuntan hacia el lado
    'H':     [0, 1, 1, 0, 0],  # índice y medio horizontales
    'I':     [0, 0, 0, 0, 1],  # solo meñique extendido
    'J':     [0, 0, 0, 0, 1],  # meñique con movimiento en J
    'K':     [1, 1, 1, 0, 0],  # pulgar, índice y medio
    'L':     [1, 1, 0, 0, 0],  # pulgar e índice en L
    'M':     [0, 0, 0, 0, 0],  # tres dedos sobre el pulgar
    'N':     [0, 0, 0, 0, 0],  # dos dedos sobre el pulgar
    'O':     [1, 1, 1, 1, 1],  # todos curvados formando O
    'P':     [1, 1, 1, 0, 0],  # índice apunta abajo
    'Q':     [1, 1, 0, 0, 0],  # índice y pulgar apuntan abajo
    'R':     [0, 1, 1, 0, 0],  # índice y medio entrelazados
    'S':     [0, 0, 0, 0, 0],  # puño con pulgar encima
    'T':     [1, 0, 0, 0, 0],  # pulgar entre índice y medio
    'U':     [0, 1, 1, 0, 0],  # índice y medio juntos arriba
    'V':     [0, 1, 1, 0, 0],  # índice y medio separados (victoria)
    'W':     [0, 1, 1, 1, 0],  # índice, medio y anular extendidos
    'X':     [0, 1, 0, 0, 0],  # índice curvado (gancho)
    'Y':     [1, 0, 0, 0, 1],  # pulgar y meñique extendidos (surfer)
    'Z':     [0, 1, 0, 0, 0],  # índice traza Z
    # Clase 27 — posibles nombres alternativos para el gesto extra:
    'del':   [0, 1, 1, 1, 0],
    'space': [1, 1, 1, 1, 1],
    'nothing': [0, 0, 0, 0, 0],
}

NUM_FINGERS = 5
print('FINGER_MAP_BY_NAME definido con', len(FINGER_MAP_BY_NAME), 'entradas.')

In [ ]:
# Nota: el heatmap completo se genera en la celda "3. Carga del dataset",
# una vez que se conocen los nombres reales de las clases (CLASS_NAMES).
# Aquí mostramos el mapeo parcial de las letras A-Z definidas en FINGER_MAP_BY_NAME.

known_keys = [k for k in FINGER_MAP_BY_NAME if len(k) == 1]  # solo letras únicas
preview_matrix = np.array([FINGER_MAP_BY_NAME[k] for k in known_keys], dtype=np.float32)

plt.figure(figsize=(7, 9))
sns.heatmap(preview_matrix, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=FINGER_NAMES, yticklabels=known_keys)
plt.title('Mapeo anatómico A–Z: clase ASL → dedos extendidos')
plt.tight_layout()
plt.show()

print('\nFrecuencia de extensión por dedo (sobre letras A–Z):')
print(pd.Series(preview_matrix.mean(axis=0), index=FINGER_NAMES).round(3))

## 3. Carga del dataset

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_path = next(DATA_DIR.glob('**/X.npy'), None) or next(DATA_DIR.glob('**/*data*.npy'), None)
Y_path = next(DATA_DIR.glob('**/Y.npy'), None) or next(DATA_DIR.glob('**/*label*.npy'), None)

if X_path is None or Y_path is None:
    all_npy = sorted(DATA_DIR.glob('**/*.npy'))
    print('Archivos disponibles:', [f.name for f in all_npy])
    X_path, Y_path = all_npy[0], all_npy[1]

X = np.load(X_path)
Y = np.load(Y_path)

# ── Convertir etiquetas a enteros ──────────────────────────────────────────
# Aplanar si Y tiene forma (N, 1)
if Y.ndim == 2 and Y.shape[1] == 1:
    Y = Y.flatten()

if Y.dtype.kind in ('U', 'S', 'O'):
    le = LabelEncoder()
    y_class = le.fit_transform(Y)
    CLASS_NAMES = le.classes_
elif Y.ndim == 2:
    y_class = np.argmax(Y, axis=1)
    CLASS_NAMES = np.array([str(i) for i in range(Y.shape[1])])
else:
    y_class = Y.astype(int)
    CLASS_NAMES = np.array([str(i) for i in np.unique(y_class)])

NUM_CLASSES = len(CLASS_NAMES)
print(f'Clases encontradas ({NUM_CLASSES}): {CLASS_NAMES}')

# ── Construir LABEL_MATRIX dinámica según CLASS_NAMES ─────────────────────
DEFAULT_FINGER = [0, 0, 0, 0, 0]  # fallback para clases no definidas en FINGER_MAP
LABEL_MATRIX = np.array(
    [FINGER_MAP_BY_NAME.get(name, DEFAULT_FINGER) for name in CLASS_NAMES],
    dtype=np.float32
)

# Mostrar clases sin mapeo definido (para revisión manual)
unmapped = [name for name in CLASS_NAMES if name not in FINGER_MAP_BY_NAME]
if unmapped:
    print(f'\n⚠ Clases sin mapeo en FINGER_MAP_BY_NAME (usarán [0,0,0,0,0]): {unmapped}')
    print('  → Actualiza FINGER_MAP_BY_NAME con la configuración correcta de dedos.')

# Heatmap del mapeo real con las clases del dataset
plt.figure(figsize=(7, max(6, NUM_CLASSES * 0.35)))
sns.heatmap(LABEL_MATRIX, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=FINGER_NAMES, yticklabels=CLASS_NAMES)
plt.title('Mapeo anatómico: clases del dataset → dedos extendidos')
plt.tight_layout()
plt.show()

# ── Mapear cada muestra a su vector de dedos ──────────────────────────────
y_fingers = LABEL_MATRIX[y_class]   # shape: (N, 5)

# ── Normalizar y aplanar imágenes ─────────────────────────────────────────
X_norm = X.astype(np.float32) / 255.0
if X_norm.ndim == 4:
    N, H, W, C = X_norm.shape
    X_flat = X_norm.reshape(N, H * W * C)
elif X_norm.ndim == 3:
    N, H, W = X_norm.shape
    X_flat = X_norm.reshape(N, H * W)
else:
    X_flat = X_norm

print(f'\nX_flat shape:    {X_flat.shape}')
print(f'y_fingers shape: {y_fingers.shape}')
print(f'Distribución por dedo (% extendido): {y_fingers.mean(axis=0).round(3)}')

## 4. División train/test

In [ ]:
X_train, X_test, y_train, y_test, yc_train, yc_test = train_test_split(
    X_flat, y_fingers, y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

INPUT_DIM = X_train.shape[1]
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Input dimension: {INPUT_DIM}')

## 5. Construcción del modelo multi-label

In [ ]:
def build_finger_model(input_dim, num_outputs=5):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_outputs, activation='sigmoid')  # sigmoid para multi-label
    ], name='DNN_FingerDetector')

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['binary_accuracy']
    )
    return model


model_fingers = build_finger_model(INPUT_DIM)
model_fingers.summary()

## 6. Entrenamiento

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, verbose=0)
]

history = model_fingers.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

## 7. Evaluación

In [ ]:
y_prob = model_fingers.predict(X_test, verbose=0)
y_pred = (y_prob >= 0.5).astype(int)

# Métricas globales
hl = hamming_loss(y_test, y_pred)
f1_micro  = f1_score(y_test, y_pred, average='micro')
f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_sample = f1_score(y_test, y_pred, average='samples', zero_division=0)

print(f'Hamming Loss:    {hl:.4f}  (menor es mejor)')
print(f'F1 micro:        {f1_micro:.4f}')
print(f'F1 macro:        {f1_macro:.4f}')
print(f'F1 by-sample:    {f1_sample:.4f}')

In [ ]:
# Métricas por dedo
finger_results = []
for i, name in enumerate(FINGER_NAMES):
    acc  = accuracy_score(y_test[:, i], y_pred[:, i])
    f1   = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    prec = f1_score(y_test[:, i], y_pred[:, i], average='binary', zero_division=0)
    n_ext = y_test[:, i].sum()
    finger_results.append({
        'Dedo': name,
        'Accuracy': acc,
        'F1': f1,
        'Muestras extendidas en test': int(n_ext)
    })

df_fingers = pd.DataFrame(finger_results).set_index('Dedo')
print('\n=== Métricas por dedo ===')
display(df_fingers.round(4))

## 8. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train')
axes[0].plot(history.history['val_loss'], label='Validación')
axes[0].set_title('Curva de pérdida — Detección de dedos')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()

axes[1].plot(history.history['binary_accuracy'],     label='Train')
axes[1].plot(history.history['val_binary_accuracy'], label='Validación')
axes[1].set_title('Curva de accuracy binario — Detección de dedos')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Binary Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Gráficas de métricas por dedo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_fingers[['Accuracy', 'F1']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Accuracy y F1 por dedo')
axes[0].set_xlabel('Dedo')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylim(0, 1)
axes[0].legend()

df_fingers['Muestras extendidas en test'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Número de muestras con el dedo extendido (test)')
axes[1].set_xlabel('Dedo')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Rendimiento y distribución por dedo', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Matrices de confusión por dedo

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, (name, ax) in enumerate(zip(FINGER_NAMES, axes)):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Flex.', 'Ext.'],
                yticklabels=['Flex.', 'Ext.'])
    ax.set_title(name)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de confusión por dedo (Flex.=flexionado, Ext.=extendido)', fontsize=11)
plt.tight_layout()
plt.show()

## 11. Visualización de predicciones en imágenes de ejemplo

In [ ]:
n_samples = 10
indices   = np.random.choice(len(X_test), n_samples, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for ax, idx in zip(axes.flatten(), indices):
    img_flat = X_test[idx]

    # Reconstruir imagen original
    try:
        if X.ndim == 4:
            _, H, W, C = X.shape
            img = img_flat.reshape(H, W, C)
        else:
            _, H, W = X.shape
            img = img_flat.reshape(H, W)
    except Exception:
        img = img_flat.reshape(int(len(img_flat)**0.5), -1)

    true_vec = y_test[idx].astype(int)
    pred_vec = y_pred[idx].astype(int)

    true_str = ''.join([n[0] for n, v in zip(FINGER_NAMES, true_vec) if v])
    pred_str = ''.join([n[0] for n, v in zip(FINGER_NAMES, pred_vec) if v])
    color    = 'green' if np.array_equal(true_vec, pred_vec) else 'red'

    if img.ndim == 3:
        ax.imshow((img * 255).astype(np.uint8))
    else:
        ax.imshow(img, cmap='gray')

    ax.set_title(f'Real: {true_str or "—"}\nPred: {pred_str or "—"}',
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Predicciones de dedos extendidos (verde=correcto, rojo=error)', fontsize=11)
plt.tight_layout()
plt.show()

## 12. Análisis de resultados

### Observaciones principales

1. **Formulación multi-label:** Cada dedo se predice de forma independiente con una salida sigmoidea propia. La pérdida `binary_crossentropy` penaliza cada dedo por separado, lo que permite que el modelo aprenda patrones independientes por dedo.

2. **Hamming Loss:** Es la fracción de etiquetas incorrectamente clasificadas. Con 5 dedos, un Hamming Loss de 0.05 significa que en promedio solo 0.25 dedos por muestra se predicen incorrectamente.

3. **Variabilidad entre dedos:**
   - El **índice** tiende a tener alta F1 porque está extendido en muchas clases (señas como D, G, J, K, L, P, Q, etc.).
   - El **anular** es el más difícil de detectar de forma independiente porque raramente se extiende solo sin el medio.
   - El **meñique** aparece solo en pocas señas (I, Y), por lo que el modelo tiene menos muestras positivas para aprenderlo.

4. **Dependencia del mapeo anatómico:** La calidad de las etiquetas depende directamente de la precisión del mapeo manual `FINGER_MAP`. Si el dataset tiene anotaciones de clase diferentes al ASL estándar, el mapeo debe ajustarse. Esto es una limitación importante del enfoque.

5. **Comparación con Punto 3:** El Punto 3 (clasificación de 27 clases) es un problema más difícil en términos de número de clases, pero el Punto 4 es más difícil conceptualmente porque requiere inferir características anatómicas de bajo nivel a partir de representaciones de alto nivel.

6. **Extensión posible:** Se podría mejorar el rendimiento usando representaciones de mediapipe (landmarks de mano 3D) en lugar de píxeles crudos, ya que los puntos de la mano codifican directamente la posición de cada articulación.